In [91]:
import numpy as np
import matplotlib.pyplot as plt
import math
import pandas as pd


In [92]:
# need to get all the hardpoints from the text file

def get_hardpoints(filename):

    #getting the key/value pairs from the text file

    coords = {}
    with open(filename, 'r') as f:
            for line in f:
                  if '=' in line:
                        key, val = line.split('=')
                        coords[key.strip()] = float(val.strip())

    return coords

In [93]:
hardpoints = get_hardpoints('hardpoints.txt')
print(hardpoints['F_UCA_OUT_X'])  # 0.366
print(hardpoints['F_UCA_OUT_Y'])  # 21.339
print(hardpoints['F_UCA_OUT_Z'])  # 10.551

#alright so this works 

-0.366
21.339
10.551


In [94]:
#FRONT CORNER
#just putting x,y,z in one

# UCA
F_UCA_OUT     = (hardpoints['F_UCA_OUT_X'],     hardpoints['F_UCA_OUT_Y'],     hardpoints['F_UCA_OUT_Z'])
F_UCA_IN_FORE = (hardpoints['F_UCA_IN_FORE_X'], hardpoints['F_UCA_IN_FORE_Y'], hardpoints['F_UCA_IN_FORE_Z'])
F_UCA_IN_AFT  = (hardpoints['F_UCA_IN_AFT_X'],  hardpoints['F_UCA_IN_AFT_Y'],  hardpoints['F_UCA_IN_AFT_Z'])

# LCA
F_LCA_OUT     = (hardpoints['F_LCA_OUT_X'],     hardpoints['F_LCA_OUT_Y'],     hardpoints['F_LCA_OUT_Z'])
F_LCA_IN_FORE = (hardpoints['F_LCA_IN_FORE_X'], hardpoints['F_LCA_IN_FORE_Y'], hardpoints['F_LCA_IN_FORE_Z'])
F_LCA_IN_AFT  = (hardpoints['F_LCA_IN_AFT_X'],  hardpoints['F_LCA_IN_AFT_Y'],  hardpoints['F_LCA_IN_AFT_Z'])

# Toe rod
F_TR_OUT = (hardpoints['F_TR_OUT_X'], hardpoints['F_TR_OUT_Y'], hardpoints['F_TR_OUT_Z'])
F_TR_IN  = (hardpoints['F_TR_IN_X'],  hardpoints['F_TR_IN_Y'],  hardpoints['F_TR_IN_Z'])

# Pushrod
F_PR_OUT = (hardpoints['F_PR_OUT_X'], hardpoints['F_PR_OUT_Y'], hardpoints['F_PR_OUT_Z'])
F_PR_IN  = (hardpoints['F_PR_IN_X'],  hardpoints['F_PR_IN_Y'],  hardpoints['F_PR_IN_Z'])





In [95]:
#REAR CORNER
#just putting x, y, z in one

# UCA
R_UCA_OUT     = (hardpoints['R_UCA_OUT_X'],     hardpoints['R_UCA_OUT_Y'],     hardpoints['R_UCA_OUT_Z'])
R_UCA_IN_FORE = (hardpoints['R_UCA_IN_FORE_X'], hardpoints['R_UCA_IN_FORE_Y'], hardpoints['R_UCA_IN_FORE_Z'])
R_UCA_IN_AFT  = (hardpoints['R_UCA_IN_AFT_X'],  hardpoints['R_UCA_IN_AFT_Y'],  hardpoints['R_UCA_IN_AFT_Z'])

# LCA
R_LCA_OUT     = (hardpoints['R_LCA_OUT_X'],     hardpoints['R_LCA_OUT_Y'],     hardpoints['R_LCA_OUT_Z'])
R_LCA_IN_FORE = (hardpoints['R_LCA_IN_FORE_X'], hardpoints['R_LCA_IN_FORE_Y'], hardpoints['R_LCA_IN_FORE_Z'])
R_LCA_IN_AFT  = (hardpoints['R_LCA_IN_AFT_X'],  hardpoints['R_LCA_IN_AFT_Y'],  hardpoints['R_LCA_IN_AFT_Z'])

# Toe rod
R_TR_OUT = (hardpoints['R_TR_OUT_X'], hardpoints['R_TR_OUT_Y'], hardpoints['R_TR_OUT_Z'])
R_TR_IN  = (hardpoints['R_TR_IN_X'],  hardpoints['R_TR_IN_Y'],  hardpoints['R_TR_IN_Z'])

# Pullrod 
R_PL_OUT = (hardpoints['R_PL_OUT_X'], hardpoints['R_PL_OUT_Y'], hardpoints['R_PL_OUT_Z'])
R_PL_IN  = (hardpoints['R_PL_IN_X'],  hardpoints['R_PL_IN_Y'],  hardpoints['R_PL_IN_Z'])

In [96]:

links_front = {
    'UCA_FORE': (F_UCA_OUT, F_UCA_IN_FORE),
    'UCA_AFT':  (F_UCA_OUT, F_UCA_IN_AFT),
    'LCA_FORE': (F_LCA_OUT, F_LCA_IN_FORE),
    'LCA_AFT':  (F_LCA_OUT, F_LCA_IN_AFT),
    'TOE_ROD':  (F_TR_OUT, F_TR_IN),
    'PUSHROD':  (F_PR_OUT, F_PR_IN),
}

links_rear = {
    'UCA_FORE': (R_UCA_OUT, R_UCA_IN_FORE),
    'UCA_AFT':  (R_UCA_OUT, R_UCA_IN_AFT),
    'LCA_FORE': (R_LCA_OUT, R_LCA_IN_FORE),
    'LCA_AFT':  (R_LCA_OUT, R_LCA_IN_AFT),
    'TOE_ROD':  (R_TR_OUT, R_TR_IN),
    'PULLROD':  (R_PL_OUT, R_PL_IN),
}

In [97]:
def compute_unit_vectors(links):
    unit_vectors = {}
    for link in links:
        outboard_point = links[link][0]
        inboard_point = links[link][1]
        
        #now need the x,y,z differences
        vx = outboard_point[0] - inboard_point[0]
        vy = outboard_point[1] - inboard_point[1]
        vz = outboard_point[2] - inboard_point[2]

        
        #getting the magnitude
        magnitude_vector = math.sqrt(vx**2 + vy**2 + vz**2)


        #calculating the unit vector 
        ux = vx/magnitude_vector
        uy = vy/magnitude_vector
        uz = vz/magnitude_vector

        unit_vectors[link] = (ux, uy,uz)

    return unit_vectors

unit_vectors_front = compute_unit_vectors(links_front)
unit_vectors_rear = compute_unit_vectors(links_rear)


In [98]:
# Tire contact patch coordinates (in)
F_CP = (0.000, 23.863, 0.0)
R_CP = (-61.000, 23.863, 0.0)

# Orion loaded rear tire radius from the 2026 brake calculations (in).
# Rear drive and inboard-brake torque both travel through the halfshaft.
REAR_TIRE_RADIUS_IN = 7.87

In [99]:
def compute_moment_arms(links, reference_point):
    moment_arms = {}
    for link in links:
        attachment_point = links[link][0] 
        rx = attachment_point[0] - reference_point[0]
        ry = attachment_point[1] - reference_point[1]
        rz = attachment_point[2] - reference_point[2]
        moment_arms[link] = (rx, ry, rz)
    return moment_arms

moment_arms_front = compute_moment_arms(links_front, F_CP)
moment_arms_rear = compute_moment_arms(links_rear, R_CP)


In [100]:
for name in moment_arms_front:
    print('FRONT', name, moment_arms_front[name])

for name in moment_arms_rear:
    print('REAR', name, moment_arms_rear[name])

FRONT UCA_FORE (-0.366, -2.524000000000001, 10.551)
FRONT UCA_AFT (-0.366, -2.524000000000001, 10.551)
FRONT LCA_FORE (0.118, -1.6980000000000004, 4.488)
FRONT LCA_AFT (0.118, -1.6980000000000004, 4.488)
FRONT TOE_ROD (2.244, -2.3279999999999994, 5.993)
FRONT PUSHROD (0.266, -3.169999999999998, 5.294)
REAR UCA_FORE (-0.1839999999999975, -3.1239999999999988, 11.6)
REAR UCA_AFT (-0.1839999999999975, -3.1239999999999988, 11.6)
REAR LCA_FORE (-0.20000000000000284, -1.1630000000000003, 4.57)
REAR LCA_AFT (-0.20000000000000284, -1.1630000000000003, 4.57)
REAR TOE_ROD (2.8130000000000024, -0.934000000000001, 9.552)
REAR PULLROD (0.5630000000000024, -4.047999999999998, 10.491)


In [101]:
def cross(a, b):
    cx = a[1]*b[2] - a[2]*b[1]
    cy = a[2]*b[0] - a[0]*b[2]
    cz = a[0]*b[1] - a[1]*b[0]
    return (cx, cy, cz)

In [102]:
def build_equilibrium_matrix(links, unit_vectors, moment_arms):
    
    # cross product
    def cross(a, b):
        cx = a[1]*b[2] - a[2]*b[1]
        cy = a[2]*b[0] - a[0]*b[2]
        cz = a[0]*b[1] - a[1]*b[0]
        return (cx, cy, cz)
    
    # list of links
    link_names = list(links.keys())
    
    # holds 6 rows
    rows = []
    
    # force balance equations (x, y, z)
    for axis in range(3):
        row = []
        for name in link_names:
            row.append(unit_vectors[name][axis])
        rows.append(row)
    
    # moment balance equations (x, y, z)
    # start with 3 empty rows to fill in
    rows.append([])
    rows.append([])
    rows.append([])
    
    for name in link_names:
        r = moment_arms[name]
        u = unit_vectors[name]
        m = cross(r, u)
        
        rows[3].append(m[0])  # x-component of moment
        rows[4].append(m[1])  # y-component of moment
        rows[5].append(m[2])  # z-component of moment
    
    # need numpy for matrix math 
    matrix = np.array(rows)
    
    return matrix, link_names

In [ ]:
def add_rear_halfshaft_torque(L_contact_patch, tire_radius_in=REAR_TIRE_RADIUS_IN):
    """The inboard rear brake and motor torque travel through
    the halfshaft, so My = Fx * loaded tire radius is included in the rigid
    wheel/upright equilibrium.  This makes the longitudinal load equivalent
    to acting at the wheel center instead of forcing the control arms to react
    the contact-patch wheel torque.
    """
    L_rear = np.asarray(L_contact_patch, dtype=float).copy()
    if L_rear.shape != (6,):
        raise ValueError(f"Rear load vector must have six components; received {L_rear.shape}")

    halfshaft_torque_lbf_in = L_rear[0] * tire_radius_in
    L_rear[4] += halfshaft_torque_lbf_in
    return L_rear, halfshaft_torque_lbf_in


def solve_load_case(A, link_names, L, label=""):
    F = np.linalg.solve(A, -L)
    print(f"--- {label} ---")
    for name, f in zip(link_names, F):
        state = "compression" if f > 0 else "tension"
        print(f"  {name}: {round(f,2)} lbf ({state})")
    return F

In [104]:
A_front, link_names_front = build_equilibrium_matrix(links_front, unit_vectors_front, moment_arms_front)
print(A_front.shape)  
print(link_names_front)

A_rear, link_names_rear = build_equilibrium_matrix(links_rear, unit_vectors_rear, moment_arms_rear)
print(A_rear.shape)  
print(link_names_rear)



(6, 6)
['UCA_FORE', 'UCA_AFT', 'LCA_FORE', 'LCA_AFT', 'TOE_ROD', 'PUSHROD']
(6, 6)
['UCA_FORE', 'UCA_AFT', 'LCA_FORE', 'LCA_AFT', 'TOE_ROD', 'PULLROD']


In [105]:
A_front

array([[-3.37827793e-01,  1.85836424e-01, -2.79755036e-01,
         2.27884531e-01, -4.71416072e-04,  7.10542266e-02],
       [ 9.26898563e-01,  9.68276005e-01,  9.55217157e-01,
         9.68765059e-01,  9.92880817e-01,  5.92203446e-01],
       [ 1.63497509e-01,  1.67052095e-01,  9.64225240e-02,
         9.77900904e-02,  1.19111128e-01, -8.02649597e-01],
       [-1.01923745e+01, -1.06379196e+01, -4.45074005e+00,
        -4.51386516e+00, -6.22762544e+00, -5.90725824e-01],
       [-3.50458095e+00,  2.02190118e+00, -1.26691846e+00,
         1.01120654e+00, -2.70110567e-01,  5.89665868e-01],
       [-1.19192222e+00,  1.14662117e-01, -3.62308427e-01,
         5.01262210e-01,  2.22692710e+00,  3.82768015e-01]])

In [106]:
A_rear

array([[ -0.7612645 ,  -0.21511748,  -0.63205289,  -0.18249938,
         -0.37917484,  -0.45984712],
       [  0.63558271,   0.95558924,   0.77110586,   0.97847482,
          0.91272858,   0.56841684],
       [  0.12849503,   0.2014291 ,   0.07684331,   0.09633796,
          0.15216103,   0.68223377],
       [ -7.77417796, -11.71409963,  -3.61332255,  -4.58367098,
         -8.86050179,  -8.72494337],
       [ -8.80702512,  -2.45829987,  -2.87311305,  -0.81475456,
         -4.04990708,  -5.20835379],
       [ -2.49513752,  -0.84785544,  -0.88929868,  -0.40794174,
          2.21335619,  -1.54144248]])

In [107]:

row_labels = ['Fx', 'Fy', 'Fz', 'Mx', 'My', 'Mz']
df_front = pd.DataFrame(A_front, index=row_labels, columns=link_names_front)
df_front

,UCA_FORE,UCA_AFT,LCA_FORE,LCA_AFT,TOE_ROD,PUSHROD
Fx,-0.337828,0.185836,-0.279755,0.227885,-0.000471,0.071054
Fy,0.926899,0.968276,0.955217,0.968765,0.992881,0.592203
Fz,0.163498,0.167052,0.096423,0.097790,0.119111,-0.802650
Mx,-10.192374,-10.637920,-4.450740,-4.513865,-6.227625,-0.590726
My,-3.504581,2.021901,-1.266918,1.011207,-0.270111,0.589666
Mz,-1.191922,0.114662,-0.362308,0.501262,2.226927,0.382768


In [108]:
df_rear = pd.DataFrame(A_rear, index=row_labels, columns=link_names_rear)
df_rear


,UCA_FORE,UCA_AFT,LCA_FORE,LCA_AFT,TOE_ROD,PULLROD
Fx,-0.761265,-0.215117,-0.632053,-0.182499,-0.379175,-0.459847
Fy,0.635583,0.955589,0.771106,0.978475,0.912729,0.568417
Fz,0.128495,0.201429,0.076843,0.096338,0.152161,0.682234
Mx,-7.774178,-11.714100,-3.613323,-4.583671,-8.860502,-8.724943
My,-8.807025,-2.458300,-2.873113,-0.814755,-4.049907,-5.208354
Mz,-2.495138,-0.847855,-0.889299,-0.407942,2.213356,-1.541442


In [109]:
L_front_lateral = np.array([0, -423.07, 290.97, 0, 0, 0])       
L_front_braking = np.array([-397.73, 0, 254.61, 0, 0, 0])        

L_rear_lateral_cp = np.array([0, -423.07, 290.97, 0, 0, 0])
L_rear_braking_cp = np.array([-397.73, 0, 254.61, 0, 0, 0])

In [110]:
F_front_lateral = solve_load_case(A_front, link_names_front, L_front_lateral, "Front - Max Lateral")
F_front_braking = solve_load_case(A_front, link_names_front, L_front_braking, "Front - Max Braking")


--- Front - Max Lateral ---
  UCA_FORE: -25.63 lbf (tension)
  UCA_AFT: -142.81 lbf (tension)
  LCA_FORE: 227.8 lbf (compression)
  LCA_AFT: 242.13 lbf (compression)
  TOE_ROD: -87.64 lbf (tension)
  PUSHROD: 371.43 lbf (compression)
--- Front - Max Braking ---
  UCA_FORE: 673.96 lbf (compression)
  UCA_AFT: -488.4 lbf (tension)
  LCA_FORE: -1444.52 lbf (tension)
  LCA_AFT: 1273.43 lbf (compression)
  TOE_ROD: -188.46 lbf (tension)
  PUSHROD: 306.5 lbf (compression)


In [111]:
L_rear_lateral, T_halfshaft_rear_lateral = add_rear_halfshaft_torque(L_rear_lateral_cp)
L_rear_braking, T_halfshaft_rear_braking = add_rear_halfshaft_torque(L_rear_braking_cp)

F_rear_lateral = solve_load_case(A_rear, link_names_rear, L_rear_lateral, "Rear - Max Lateral")
print(f"  Halfshaft torque: {T_halfshaft_rear_lateral:.2f} lbf-in")
F_rear_braking = solve_load_case(A_rear, link_names_rear, L_rear_braking, "Rear - Max Braking")
print(f"  Halfshaft torque: {T_halfshaft_rear_braking:.2f} lbf-in")

--- Rear - Max Lateral ---
  UCA_FORE: 330.32 lbf (compression)
  UCA_AFT: -35.41 lbf (tension)
  LCA_FORE: -195.62 lbf (tension)
  LCA_AFT: 713.55 lbf (compression)
  TOE_ROD: 20.62 lbf (compression)
  PULLROD: -561.58 lbf (tension)
  Halfshaft torque: 0.00 lbf-in
--- Rear - Max Braking ---
  UCA_FORE: -19.78 lbf (tension)
  UCA_AFT: 566.08 lbf (compression)
  LCA_FORE: -382.08 lbf (tension)
  LCA_AFT: 264.19 lbf (compression)
  TOE_ROD: -242.35 lbf (tension)
  PULLROD: -476.83 lbf (tension)
  Halfshaft torque: -3130.14 lbf-in


In [112]:
# import numpy as np

# traction_limit = 450  # lbf, approx max combined Fx/Fy grip

# # sweep angle around the traction circle (0 to 360 degrees)
# angle_range = np.linspace(0, 2*np.pi, 36, endpoint=False)  # 36 directions, every 10 degrees

# # sweep how close to the limit (partial grip usage, e.g. 50%, 75%, 100%)
# magnitude_fraction_range = np.array([0.5, 0.75, 1.0])

# fz_range = np.arange(-450, 500, 50)   


# def sweep_worst_case(A, link_names):
#     n = len(link_names)
#     max_comp = np.full(n, -np.inf)
#     max_tens = np.full(n, np.inf)
#     case_comp = [None] * n
#     case_tens = [None] * n

#     for angle in angle_range:
#         for frac in magnitude_fraction_range:
#             fx = traction_limit * frac * np.cos(angle)
#             fy = traction_limit * frac * np.sin(angle)

#             for fz in fz_range:
#                 L = np.array([fx, fy, fz, 0, 0, 0])
#                 F = np.linalg.solve(A, -L)

#                 for i in range(n):
#                     if F[i] > max_comp[i]:
#                         max_comp[i] = F[i]
#                         case_comp[i] = L.copy()
#                     if F[i] < max_tens[i]:
#                         max_tens[i] = F[i]
#                         case_tens[i] = L.copy()

#     return max_comp, max_tens, case_comp, case_tens

In [113]:
FZ_MAX_REAR = 300.0
FZ_MAX_FRONT = FZ_MAX_REAR * (0.4835 / 0.5165)
FZ_STEP = 1.0

MU_X = 1.56
MU_Y = 1.45

FX_CAP = 450.0
FY_CAP = 450.0

FORCE_TOL = 1e-8

def sweep_worst_case(A, link_names, fz_max, rear_halfshaft=False, tire_radius_in=0.0):
    A = np.asarray(A, dtype=float)
    n = len(link_names)

    if A.shape != (6, 6):
        raise ValueError(f"A must be 6x6 for this rigid-corner solver; received {A.shape}")

    if n != 6:
        raise ValueError(f"Expected six link names; received {n}")

    if not np.all(np.isfinite(A)):
        raise ValueError("A contains NaN or infinite values")

    rank = np.linalg.matrix_rank(A)
    if rank != 6:
        raise np.linalg.LinAlgError(f"Equilibrium matrix is singular: rank(A) = {rank}, expected 6")

    response = np.linalg.solve(A, -np.eye(6))

    max_comp = np.full(n, np.nan)
    max_tens = np.full(n, np.nan)

    case_comp = [None] * n
    case_tens = [None] * n

    def solve_and_check(L):
        F = response @ L
        residual = A @ F + L
        tolerance = 1e-9 * max(1.0, np.linalg.norm(L, ord=np.inf))
        if np.linalg.norm(residual, ord=np.inf) > tolerance:
            raise ArithmeticError(f"Static-equilibrium residual exceeded tolerance: {np.linalg.norm(residual, ord=np.inf):.3e}")
        return F

    fz_range = np.arange(0.0, fz_max + 0.5 * FZ_STEP, FZ_STEP)

    for fz in fz_range:
        fx_limit = min(MU_X * fz, FX_CAP)
        fy_limit = min(MU_Y * fz, FY_CAP)

        for i in range(n):
            # For the rear axle, Fx also creates the balancing halfshaft My.
            # Include that coupled response when selecting the governing tire-force direction.
            cx = response[i, 0]
            if rear_halfshaft:
                cx += tire_radius_in * response[i, 4]
            cy = response[i, 1]

            direction_scale = np.hypot(cx * fx_limit, cy * fy_limit)

            if direction_scale > 1e-14:
                fx_governing = fx_limit**2 * cx / direction_scale
                fy_governing = fy_limit**2 * cy / direction_scale
            else:
                fx_governing = 0.0
                fy_governing = 0.0

            L_comp = np.array([fx_governing, fy_governing, fz, 0.0, 0.0, 0.0])
            L_tens = np.array([-fx_governing, -fy_governing, fz, 0.0, 0.0, 0.0])

            if rear_halfshaft:
                L_comp, _ = add_rear_halfshaft_torque(L_comp, tire_radius_in)
                L_tens, _ = add_rear_halfshaft_torque(L_tens, tire_radius_in)

            F_comp = solve_and_check(L_comp)
            F_tens = solve_and_check(L_tens)

            if (
                F_comp[i] > FORCE_TOL
                and (np.isnan(max_comp[i]) or F_comp[i] > max_comp[i])
            ):
                max_comp[i] = F_comp[i]
                case_comp[i] = L_comp.copy()

            if (
                F_tens[i] < -FORCE_TOL
                and (np.isnan(max_tens[i]) or F_tens[i] < max_tens[i])
            ):
                max_tens[i] = F_tens[i]
                case_tens[i] = L_tens.copy()

    return max_comp, max_tens, case_comp, case_tens

In [114]:
max_comp_front, max_tens_front, case_comp_front, case_tens_front = sweep_worst_case(A_front, link_names_front, FZ_MAX_FRONT)

results_front_simple = pd.DataFrame({
    'Link': link_names_front,
    'Max Compression': [round(v, 2) for v in max_comp_front],
    'Max Tension': [round(v, 2) for v in max_tens_front],
})
results_front_simple

,Link,Max Compression,Max Tension
0,UCA_FORE,750.27,-600.29
1,UCA_AFT,659.12,-566.43
2,LCA_FORE,1368.25,-1634.10
3,LCA_AFT,1459.36,-1848.29
4,TOE_ROD,229.28,-224.88
5,PUSHROD,358.78,NaN


In [115]:
def scaled_condition_number(A, reference_length):
    A_scaled = A.copy()
    A_scaled[3:6, :] = A_scaled[3:6, :] / reference_length
    return np.linalg.cond(A_scaled)

reference_length_front = 23.863
reference_length_rear = 23.863

print("Front - raw condition number:   ", np.linalg.cond(A_front))
print("Front - scaled condition number:", scaled_condition_number(A_front, reference_length_front))

print("Rear - raw condition number:   ", np.linalg.cond(A_rear))
print("Rear - scaled condition number:", scaled_condition_number(A_rear, reference_length_rear))

Front - raw condition number:    93.73045035149866
Front - scaled condition number: 37.094317412721566
Rear - raw condition number:    110.76560938882795
Rear - scaled condition number: 31.97648516093464


In [116]:
max_comp_rear, max_tens_rear, case_comp_rear, case_tens_rear = sweep_worst_case(
    A_rear,
    link_names_rear,
    FZ_MAX_REAR,
    rear_halfshaft=True,
    tire_radius_in=REAR_TIRE_RADIUS_IN,
)

results_rear_simple = pd.DataFrame({
    'Link': link_names_rear,
    'Max Compression': [round(v, 2) for v in max_comp_rear],
    'Max Tension': [round(v, 2) for v in max_tens_rear],
    'Compression Halfshaft Torque (lbf-in)': [
        round(case[0] * REAR_TIRE_RADIUS_IN, 2) if case is not None else np.nan
        for case in case_comp_rear
    ],
    'Tension Halfshaft Torque (lbf-in)': [
        round(case[0] * REAR_TIRE_RADIUS_IN, 2) if case is not None else np.nan
        for case in case_tens_rear
    ],
})

results_rear_simple

,Link,Max Compression,Max Tension,Compression Halfshaft Torque (lbf-in),Tension Halfshaft Torque (lbf-in)
0,UCA_FORE,513.06,-39.49,3296.21,-3307.67
1,UCA_AFT,797.11,-178.84,-2491.59,2538.16
2,LCA_FORE,628.19,-496.88,3118.48,-3118.48
3,LCA_AFT,848.74,-1214.29,-1628.81,1628.81
4,TOE_ROD,265.59,-275.69,3525.95,-3524.75
5,PULLROD,NaN,-579.41,NaN,-715.32


In [117]:
E_PSI = 29_700_000
YIELD_STRENGTH_PSI = 50_000

front_tube_properties = {
    'UCA_FORE': {'A': 0.257708772, 'I': 0.007298393, 'E': E_PSI, 'yield': YIELD_STRENGTH_PSI},
    'UCA_AFT':  {'A': 0.064873888, 'I': 0.002832759, 'E': E_PSI, 'yield': YIELD_STRENGTH_PSI},
    'LCA_FORE': {'A': 0.107910566, 'I': 0.006660807, 'E': E_PSI, 'yield': YIELD_STRENGTH_PSI},
    'LCA_AFT':  {'A': 0.107910566, 'I': 0.006660807, 'E': E_PSI, 'yield': YIELD_STRENGTH_PSI},
    'TOE_ROD':  {'A': 0.088668311, 'I': 0.003703864, 'E': E_PSI, 'yield': YIELD_STRENGTH_PSI},
    'PUSHROD':  {'A': 0.088668311, 'I': 0.003703864, 'E': E_PSI, 'yield': YIELD_STRENGTH_PSI},
}

rear_tube_properties = {
    'UCA_FORE': {'A': 0.064873888, 'I': 0.002832759, 'E': E_PSI, 'yield': YIELD_STRENGTH_PSI},
    'UCA_AFT':  {'A': 0.064873888, 'I': 0.002832759, 'E': E_PSI, 'yield': YIELD_STRENGTH_PSI},
    'LCA_FORE': {'A': 0.107910566, 'I': 0.006660807, 'E': E_PSI, 'yield': YIELD_STRENGTH_PSI},
    'LCA_AFT':  {'A': 0.107910566, 'I': 0.006660807, 'E': E_PSI, 'yield': YIELD_STRENGTH_PSI},
    'TOE_ROD':  {'A': 0.088668311, 'I': 0.003703864, 'E': E_PSI, 'yield': YIELD_STRENGTH_PSI},
    'PULLROD':  {'A': 0.088668311, 'I': 0.003703864, 'E': E_PSI, 'yield': YIELD_STRENGTH_PSI},
}

In [118]:
#Stress = Force / Area
#Critical Buckling Load = π² × E × I / L²

In [119]:
FORCE_TOL = 1e-8

def compute_length(links):
    lengths = {}
    for name in links:
        o, i = links[name]
        lengths[name] = math.sqrt((o[0]-i[0])**2 + (o[1]-i[1])**2 + (o[2]-i[2])**2)
    return lengths

def tube_area_and_I(props):
    if 'A' in props and 'I' in props:
        return props['A'], props['I']
    OD = props['OD']
    t = props['t']
    if OD <= 0 or t <= 0 or 2*t >= OD:
        raise ValueError(f"Invalid tube dimensions: OD={OD}, t={t}")
    ID = OD - 2*t
    A = (math.pi/4) * (OD**2 - ID**2)
    I = (math.pi/64) * (OD**4 - ID**4)
    return A, I

def structural_check(link_names, forces, links, tube_properties):
    if len(link_names) != len(forces):
        raise ValueError("link_names and forces must have equal length")

    lengths = compute_length(links)
    results = []
    
    for name, F in zip(link_names, forces):
        if name not in links or name not in tube_properties:
            raise KeyError(f"Missing geometry or properties for {name}")

        props = tube_properties[name]
        A, I = tube_area_and_I(props)
        L = lengths[name]
        E = props['E']
        yield_strength = props['yield']
        K = props.get('K', 1.0)

        if L <= 0 or A <= 0 or I <= 0:
            raise ValueError(f"{name}: L, A, and I must be positive")
        if E <= 0 or yield_strength <= 0 or K <= 0:
            raise ValueError(f"{name}: E, yield strength, and K must be positive")

        effective_length = K * L
        r = math.sqrt(I / A)
        slenderness = effective_length / r

        if not np.isfinite(F) or abs(F) <= FORCE_TOL:
            results.append({
                'Link': name,
                'Force (lbf)': 'N/A',
                'Stress (psi)': 'N/A',
                'Yield Margin': 'N/A',
                'Buckling Method': 'N/A',
                'KL/r': round(slenderness, 1),
                'Buckling Load (lbf)': 'N/A',
                'Buckling Margin': 'N/A',
            })
            continue

        stress = F / A
        yield_margin = yield_strength / abs(stress)

        if F > FORCE_TOL:
            cc = math.sqrt(2 * math.pi**2 * E / yield_strength)
            if slenderness <= cc:
                buckling_method = 'Johnson'
                critical_stress = yield_strength * (1.0 - yield_strength * slenderness**2 / (4.0 * math.pi**2 * E))
            else:
                buckling_method = 'Euler'
                critical_stress = math.pi**2 * E / slenderness**2
            Pcr = A * critical_stress
            buckling_margin = Pcr / F
        else:
            buckling_method = None
            Pcr = None
            buckling_margin = None

        results.append({
            'Link': name, 
            'Force (lbf)': round(F,1), 
            'Stress (psi)': round(stress,1),
            'Yield Margin': round(yield_margin,2),
            'Buckling Method': buckling_method if buckling_method is not None else 'N/A',
            'KL/r': round(slenderness, 1),
            'Buckling Load (lbf)': round(Pcr,1) if Pcr is not None else 'N/A',
            'Buckling Margin': round(buckling_margin,2) if buckling_margin is not None else 'N/A',
        })
        
    return pd.DataFrame(results)

In [120]:
print("Front - Compression:")
display(structural_check(link_names_front, max_comp_front, links_front, front_tube_properties))
print("Front - Tension:")
display(structural_check(link_names_front, max_tens_front, links_front, front_tube_properties))

print("Rear - Compression:")
display(structural_check(link_names_rear, max_comp_rear, links_rear, rear_tube_properties))
print("Rear - Tension:")
display(structural_check(link_names_rear, max_tens_rear, links_rear, rear_tube_properties))

Front - Compression:


,Link,Force (lbf),Stress (psi),Yield Margin,Buckling Method,KL/r,Buckling Load (lbf),Buckling Margin
0,UCA_FORE,750.3,2911.3,17.17,Johnson,76.8,9644.8,12.86
1,UCA_AFT,659.1,10160.1,4.92,Johnson,59.6,2752.1,4.18
2,LCA_FORE,1368.3,12679.5,3.94,Johnson,55.9,4677.8,3.42
3,LCA_AFT,1459.4,13523.8,3.70,Johnson,55.1,4697.7,3.22
4,TOE_ROD,229.3,2585.8,19.34,Johnson,62.3,3700.3,16.14
5,PUSHROD,358.8,4046.3,12.36,Johnson,57.6,3805.4,10.61


Front - Tension:


,Link,Force (lbf),Stress (psi),Yield Margin,Buckling Method,KL/r,Buckling Load (lbf),Buckling Margin
0,UCA_FORE,-600.3,-2329.3,21.47,N/A,76.8,N/A,N/A
1,UCA_AFT,-566.4,-8731.2,5.73,N/A,59.6,N/A,N/A
2,LCA_FORE,-1634.1,-15143.1,3.3,N/A,55.9,N/A,N/A
3,LCA_AFT,-1848.3,-17128.0,2.92,N/A,55.1,N/A,N/A
4,TOE_ROD,-224.9,-2536.2,19.71,N/A,62.3,N/A,N/A
5,PUSHROD,N/A,N/A,N/A,N/A,57.6,N/A,N/A


Rear - Compression:


,Link,Force (lbf),Stress (psi),Yield Margin,Buckling Method,KL/r,Buckling Load (lbf),Buckling Margin
0,UCA_FORE,513.1,7908.6,6.32,Johnson,68.0,2603.3,5.07
1,UCA_AFT,797.1,12287.1,4.07,Johnson,47.9,2926.4,3.67
2,LCA_FORE,628.2,5821.4,8.59,Johnson,60.2,4560.7,7.26
3,LCA_AFT,848.7,7865.2,6.36,Johnson,47.5,4877.2,5.75
4,TOE_ROD,265.6,2995.3,16.69,Johnson,61.8,3712.1,13.98
5,PULLROD,N/A,N/A,N/A,N/A,51.3,N/A,N/A


Rear - Tension:


,Link,Force (lbf),Stress (psi),Yield Margin,Buckling Method,KL/r,Buckling Load (lbf),Buckling Margin
0,UCA_FORE,-39.5,-608.7,82.14,N/A,68.0,N/A,N/A
1,UCA_AFT,-178.8,-2756.7,18.14,N/A,47.9,N/A,N/A
2,LCA_FORE,-496.9,-4604.5,10.86,N/A,60.2,N/A,N/A
3,LCA_AFT,-1214.3,-11252.7,4.44,N/A,47.5,N/A,N/A
4,TOE_ROD,-275.7,-3109.2,16.08,N/A,61.8,N/A,N/A
5,PULLROD,-579.4,-6534.6,7.65,N/A,51.3,N/A,N/A
